In [ ]:
# Imports and setup
import json
import os
import sys
from typing import List, Optional, Dict
import gc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import auc, roc_curve
from tqdm import tqdm

# Import memory optimization helpers
sys.path.append('..')
try:
    from memory_optimized_notebook import load_dataframe, clear_plt, print_memory_usage
except ImportError:
    # Fallback if module not available
    def print_memory_usage(): pass
    def clear_plt(): 
        plt.close('all')
        gc.collect()
    load_dataframe = pd.read_csv

# Force garbage collection at start
gc.collect()

sys.path.append('..')
sys.path.append('../..')
from plotting_notebooks.plotting_utils import (
    load_in_combined_results,
    load_monitoring_data,
    plot_markov_analysis,
    plot_roc_curves,
    plot_suspiciousness_distributions,
    plot_cost_vs_safety_curves,
    get_roc_curve,
    interpolate_tpr,
    prepare_monitoring_data,
    merge_monitoring_with_combined_results,
    filter_and_prepare_data,
    is_in_training_data,
    plot_combined_cost_vs_safety_curve,
    plot_combined_cost_vs_tpr_curve,
)
from simulate_markov_utils import plot_markov_df
from utils.analysis_utils import extract_verdict, balance_classes_with_oversampling
from utils.path_utils import resolve_path

# Add memory tracking
print_memory_usage()


In [ ]:
# Constants and configuration
MAIN_AGENT_MODELS = ['claude-3-5-sonnet-20241022']
MONITOR_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022', 'o3-mini', 'o1', 'claude-3-5-haiku-20241022']
DEFAULT_TRANSCRIPT_TYPES = ["transcripts", "transcripts_no_hint", "transcripts_no_secondary_task"]
DOWNSAMPLE_FACTORS = [2, 3, 5, 10, 15]
BACK_FRACS = [0.1, 0.2, 0.5]
NUM_MONITORS = 10

# Step mapping for fine-tuned models
FT_STEP_MAPPING = {
    "gpt-4o-2024-08-06": 0,
    'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFx2l:ckpt-step-70': 70,
    # 'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFtC0:ckpt-step-140': 140,
    # 'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFa40': 210,
    # 'ft:gpt-4o-2024-08-06:scale-safety-research-1:unbalanced-data:BDXppMow:ckpt-step-56': 56,
    # 'ft:gpt-4o-2024-08-06:scale-safety-research-1:unbalanced-data:BDXpp5YL:ckpt-step-112': 112,
    # 'ft:gpt-4o-2024-08-06:scale-safety-research-1:unbalanced-data:BDXppDFa': 168,
}
FT_MONITOR_MODELS = list(FT_STEP_MAPPING.keys())

# Cache for combined_results (modified to be more memory-efficient)
_combined_results_cache = {}


In [ ]:
# Helper functions for data loading

def load_combined_results(include_failed_attempts: bool, include_failed_main_task: bool) -> pd.DataFrame:
    """Load and cache combined_results."""
    cache_key = f"{include_failed_attempts}_{include_failed_main_task}"
    if cache_key in _combined_results_cache:
        print(f"Using cached combined_results for {cache_key}")
        combined_results = _combined_results_cache[cache_key]
    else:
        combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)
        _combined_results_cache[cache_key] = combined_results
    
    # Add memory tracking
    print_memory_usage()
    combined_results = combined_results[combined_results.transcript_type.str.contains('v1')]
    combined_results['transcript_type'] = combined_results['transcript_type'].str.replace('_v1', '')
    return combined_results
combined_results = load_combined_results(include_failed_attempts=False, include_failed_main_task=False)
combined_results = combined_results[combined_results.main_agent.isin(MAIN_AGENT_MODELS)]
combined_results.transcript_type.value_counts()

In [ ]:
# Functions for loading fine-tuned monitoring data
def prepare_ft_monitoring_data(base_dir: str, monitor_models: List[str] = FT_MONITOR_MODELS) -> pd.DataFrame:
    """Load and prepare the fine-tuned monitoring data."""
    # Use the generalized function from plotting_utils
    ft_monitoring_df = prepare_monitoring_data(
        base_dir=base_dir,
        monitor_models=monitor_models,
        num_monitors=NUM_MONITORS,
        compute_aggregates=True,
        tag_word='verdict'
    )
    ft_monitoring_df = ft_monitoring_df[ft_monitoring_df['usage_8'] != {}]
    # Add side_task_evaluation column
    # ft_monitoring_df['side_task_evaluation'] = ft_monitoring_df.transcript_type != 'transcripts_no_secondary_task'
    for i in range(1, 11):
        ft_monitoring_df[f'mean_suspiciousness_first_{i}'] = ft_monitoring_df[[f'suspiciousness_{j}' for j in range(1, i+1)]].mean(axis=1)
        # ft_monitoring_df[f'max_suspiciousness_first_{i}'] = ft_monitoring_df[[f'suspiciousness_{j}' for j in range(1, i+1)]].max(axis=1)
        # ft_monitoring_df[f'min_suspiciousness_first_{i}'] = ft_monitoring_df[[f'suspiciousness_{j}' for j in range(1, i+1)]].min(axis=1)
        # # Calculate usage for each mean_suspiciousness_first_i by summing the costs
        usage_column = ft_monitoring_df.apply(
            lambda row: sum([row[f'usage_{j}'][0]['cost_usd'] for j in range(1, i+1) if len(row[f'usage_{j}']) > 0]),
            axis=1
        )
        ft_monitoring_df[f'mean_suspiciousness_first_{i}_usage'] = usage_column
        ft_monitoring_df[f'max_suspiciousness_first_{i}_usage'] = usage_column
        ft_monitoring_df[f'min_suspiciousness_first_{i}_usage'] = usage_column
        ft_monitoring_df[f'{i}th_highest_suspiciousness_usage'] = usage_column
    
    # Add memory tracking
    print_memory_usage()
    return ft_monitoring_df

In [ ]:
combined_results.side_task_evaluation.unique()

In [ ]:
# Functions for loading turn-by-turn monitoring data

def get_downsampled_responses(x, downsample_factor, back_frac=1.0):
    """Extract responses with specified downsampling factor."""
    all_indices = list(x.values())[::downsample_factor] + [list(x.values())[-1]]
    back_indices = all_indices[-int(len(all_indices) * back_frac):]
    return back_indices

def prepare_turn_by_turn_data(base_dir: str, 
                             downsample_factors: List[int] = DOWNSAMPLE_FACTORS, 
                             thresholds_to_consider = [4, 9, 14, 19, 29, 39], 
                             nth_highest_indices: List[int] = [2, 3, 5], 
                             suffix: str = 'turn_by_turn') -> pd.DataFrame:
    """Load and prepare the turn-by-turn monitoring data."""
    # Load monitoring data
    turn_by_turn_df = load_monitoring_data(base_dir=base_dir)
    
    # Calculate downsampled max suspiciousness scores
    for downsample_factor in downsample_factors:
        turn_by_turn_df[f'max_suspiciousness_{downsample_factor}_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(max([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 
                               for y in get_downsampled_responses(x, downsample_factor)]))
        )
        turn_by_turn_df[f'max_suspiciousness_usage_{downsample_factor}_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, downsample_factor)]))
        )
    
    # Force garbage collection after intensive processing
    gc.collect()
    
    for back_frac in BACK_FRACS:
        turn_by_turn_df[f'max_suspiciousness_back_{back_frac}_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(max([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 
                               for y in get_downsampled_responses(x, 1, back_frac)]))
        )
        turn_by_turn_df[f'max_suspiciousness_back_{back_frac}_usage_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, 1, back_frac)]))
        )
    gc.collect()
        
    # Calculate fraction above thresholds
    for threshold in thresholds_to_consider:
        turn_by_turn_df[f'frac_above_{threshold}_{suffix}'] = 100 * turn_by_turn_df.data.apply(
            lambda x: float(np.mean([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') > threshold or 0.0 
                                   for y in get_downsampled_responses(x, 1)]))
        )

        turn_by_turn_df[f'frac_above_{threshold}_usage_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, 1)]))
        )
    
    # Force garbage collection after intensive processing
    gc.collect()
    
    # Calculate nth highest suspiciousness scores
    for n in nth_highest_indices:
        turn_by_turn_df[f'{n}th_highest_suspiciousness_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: sorted([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 
                            for y in list(x.values())], reverse=True)[n-1] if len(x) >= n else 0.0
        )

        turn_by_turn_df[f'{n}th_highest_suspiciousness_usage_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, 1)]))
        )
    
    # Calculate final suspiciousness and usage
    turn_by_turn_df[f'final_suspiciousness_{suffix}'] = turn_by_turn_df.data.apply(
        lambda x: float(extract_verdict(x[str(max([int(k) for k in x.keys()]))]["content"] 
                                      if isinstance(x[str(max([int(k) for k in x.keys()]))], dict) 
                                      and "content" in x[str(max([int(k) for k in x.keys()]))] 
                                      else x[str(max([int(k) for k in x.keys()]))], tag_word='verdict'))
    )
    
    turn_by_turn_df[f'final_usage_{suffix}'] = turn_by_turn_df.data.apply(
        lambda x: float(x[str(max([int(k) for k in x.keys()]))]["usage"][0]["cost_usd"])
    )
    
    # Add side_task_evaluation column
    # turn_by_turn_df['side_task_evaluation'] = turn_by_turn_df.transcript_type != 'transcripts_no_secondary_task'
    
    # Memory tracking and cleanup
    print_memory_usage()
    gc.collect()
    
    return turn_by_turn_df


In [ ]:
# Main function to load all data

def load_all_monitoring_data(include_failed_attempts=False, include_failed_main_task=False):
    """Load all monitoring data required for combined analysis."""
    # Load base results
    merge_on = ['task_name', 'transcript_type', 'main_agent', 'file_name']
    combined_results = load_combined_results(include_failed_attempts, include_failed_main_task)
    combined_results = combined_results[combined_results.main_agent.isin(MAIN_AGENT_MODELS)]
    gc.collect()  # Force cleanup after loading large dataset
    
    # Load fine-tuned monitoring data for system prompt 1 and 5
    ft_monitoring_sp1 = prepare_ft_monitoring_data(base_dir='monitor_responses_v1/system_prompt_1_temp_0')
    ft_monitoring_sp1 = ft_monitoring_sp1[ft_monitoring_sp1.main_agent.isin(MAIN_AGENT_MODELS)]
    ft_monitoring_sp1 = pd.merge(ft_monitoring_sp1, combined_results[merge_on + ['main_task_success', 'side_task_evaluation']].groupby(merge_on).first().reset_index(), on=merge_on, how = 'left').dropna()
    ft_monitoring_sp1['side_task_evaluation'] = ft_monitoring_sp1['side_task_evaluation'].astype(bool)
    gc.collect()  # Force cleanup after loading large dataset
    
    ft_monitoring_sp5 = prepare_ft_monitoring_data(base_dir='monitor_responses_v1/system_prompt_5_temp_0')
    ft_monitoring_sp5 = ft_monitoring_sp5[ft_monitoring_sp5.main_agent.isin(MAIN_AGENT_MODELS)]
    ft_monitoring_sp5 = pd.merge(ft_monitoring_sp5, combined_results[merge_on + ['main_task_success', 'side_task_evaluation']].groupby(merge_on).first().reset_index(), on=merge_on, how = 'left').dropna()
    ft_monitoring_sp5['side_task_evaluation'] = ft_monitoring_sp5['side_task_evaluation'].astype(bool)
    gc.collect()  # Force cleanup after loading large dataset
    
    # Load turn-by-turn monitoring data
    turn_by_turn_df = prepare_turn_by_turn_data(base_dir='monitor_responses_v1/turn_by_turn_monitoring')
    turn_by_turn_df = turn_by_turn_df[turn_by_turn_df.main_agent.isin(MAIN_AGENT_MODELS)]
    turn_by_turn_df = pd.merge(turn_by_turn_df, combined_results[merge_on + ['main_task_success', 'side_task_evaluation']].groupby(merge_on).first().reset_index(), on=merge_on, how = 'left').dropna()
    turn_by_turn_df['side_task_evaluation'] = turn_by_turn_df['side_task_evaluation'].astype(bool)
    gc.collect()  # Force cleanup after loading large dataset
    
    # Load training data for reference
    training_data_path = resolve_path('finetuning/training_data_dfs/combined_3.csv')
    training_data = pd.read_csv(training_data_path)
    gc.collect()  # Force cleanup after loading large dataset
    
    # Tag which examples are in training data
    for df in [ft_monitoring_sp1, ft_monitoring_sp5, turn_by_turn_df]:
        df['in_training_data'] = df.apply(
            lambda row: is_in_training_data(row, training_data), 
            axis=1
        )
    
    # Memory tracking and cleanup
    print_memory_usage()
    gc.collect()
    
    return combined_results, ft_monitoring_sp1, ft_monitoring_sp5, turn_by_turn_df, training_data


In [ ]:
# Example usage
print("Loading all monitoring data...")
combined_results, ft_sp1, ft_sp5, turn_by_turn, training_data = load_all_monitoring_data()
print("Data loaded successfully!")
print(f"FT System Prompt 1 shape: {ft_sp1.shape}")
print(f"FT System Prompt 5 shape: {ft_sp5.shape}")
print(f"Turn-by-turn shape: {turn_by_turn.shape}")

In [ ]:
len(ft_sp1)

In [ ]:
# Clear unused references to free memory
gc.collect()
print_memory_usage()


In [ ]:
# unstacked = ft_sp5[ft_sp5.task_name.isin(['election_tampering'])].drop(columns=['in_training_data', 'data'] + [f'response_{i}' for i in range(1, 11)]).set_index(['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name', 'side_task_evaluation']).unstack(level='monitor_agent')
# columns = unstacked.columns
# # Create a list to store all rows
# averaging_results_list = []
# for i, column_1 in enumerate(columns):
#     print(i/len(columns))
#     if 'usage' in column_1[0] or column_1[0] in [f'suspiciousness_{i}' for i in range(1, 11)]:
#         continue
#     for j, column_2 in enumerate(columns[i:]):
#         if 'usage' in column_2[0] or column_2[0] in [f'suspiciousness_{i}' for i in range(1, 11)]:
#             continue
#         unstacked[f'ave'] = (unstacked[column_1] + unstacked[column_2]) / 2
#         fpr, tpr, thresholds = roc_curve(unstacked.index.get_level_values('side_task_evaluation'), unstacked[f'ave'])
#         auc_score_ave = auc(fpr, tpr)
#         fpr, tpr, thresholds = roc_curve(unstacked.index.get_level_values('side_task_evaluation'), unstacked[column_1])
#         auc_score_1 = auc(fpr, tpr)
#         fpr, tpr, thresholds = roc_curve(unstacked.index.get_level_values('side_task_evaluation'), unstacked[column_2])
#         auc_score_2 = auc(fpr, tpr)
#         # Create a new row as a dictionary and add to list
#         averaging_results_list.append({
#             'column_1': column_1,
#             'column_2': column_2, 
#             'ave_auc': auc_score_ave,
#             'column_1_auc': auc_score_1,
#             'column_2_auc': auc_score_2
#         })
#     # Create DataFrame once at the end
# averaging_results_df = pd.DataFrame(averaging_results_list)
# display(averaging_results_df)
# #%%
# averaging_results_df['improvement'] = averaging_results_df['ave_auc'] - np.maximum(averaging_results_df['column_1_auc'], averaging_results_df['column_2_auc'])
# averaging_results_df = averaging_results_df.sort_values(by='ave_auc', ascending=False)
# averaging_results_df.column_1 = averaging_results_df.column_1.apply(lambda x: (x[0], FT_STEP_MAPPING[x[1]]))
# averaging_results_df.column_2 = averaging_results_df.column_2.apply(lambda x: (x[0], FT_STEP_MAPPING[x[1]]))
# #%%
# gc.collect()
# averaging_results_df.sort_values(by='ave_auc', ascending=False).head(20)

In [ ]:
# plot_roc_curves(turn_by_turn[turn_by_turn.task_name == 'charity_allocation'], combined_results, give_hint=None, include_failed_attempts=False, include_failed_main_task=False, monitors=turn_by_turn.monitor_agent.unique(), metrics=[f'max_suspiciousness_{i}_turn_by_turn' for i in range(1, 15)] + ['final_suspiciousness_turn_by_turn'] + [f'{i}th_highest_suspiciousness_turn_by_turn' for i in [2,3,5]])

In [ ]:
# ft_model = FT_MONITOR_MODELS[2]
# turn_by_turn_model = 'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFx2l:ckpt-step-70'

# filtered_ft_sp1 = ft_sp1[ft_sp1.monitor_agent == ft_model]
# filtered_ft_sp5 = ft_sp5[ft_sp5.monitor_agent == ft_model]
# filtered_turn_by_turn = turn_by_turn[turn_by_turn.monitor_agent == turn_by_turn_model]


# gc.collect()
# print_memory_usage()
# #%%
# ft_sp1.monitor_agent.unique()

In [ ]:
# Merge filtered_ft_sp1 and filtered_turn_by_turn
# print("Merging dataframes...")
# merged_ft_turn = pd.merge(
#     filtered_ft_sp1,
#     filtered_turn_by_turn,
#     on=['task_name', 'transcript_type', 'main_agent', 'file_name', 'side_task_evaluation'],
#     suffixes=('_ft', '_tbt'),
#     how='inner'
# )
# print(f"Merged shape: {merged_ft_turn.shape}")


# gc.collect()
# print_memory_usage()

# merged_ft_turn = merged_ft_turn[~merged_ft_turn.task_name.isin(training_data.task_name)]
# # merged_ft_turn = merged_ft_turn[~merged_ft_turn.task_name.isin(['election_tampering', 'charity_allocation'])]
# print("length of merged_ft_turn", len(merged_ft_turn))
# for metric in ['max_suspiciousness', 'mean_suspiciousness',  'final_suspiciousness_turn_by_turn'] + [f'{i}th_highest_suspiciousness' for i in range(1, 6)] + [f'max_suspiciousness_{i}_turn_by_turn' for i in DOWNSAMPLE_FACTORS]:
#     fpr, tpr, thresholds = roc_curve(merged_ft_turn.side_task_evaluation, merged_ft_turn[metric])
#     auc_score = auc(fpr, tpr)
#     print(f"AUC score {metric}: {round(auc_score, 3)}")
# gc.collect()
# #%%
# print("Notebook completed successfully!")
# # Clear all plots at the end
# clear_plt()


In [ ]:
# Reload imports after changes
from importlib import reload
import plotting_notebooks.plotting_utils as plotting_utils
import simulate_markov_utils
import utils.analysis_utils as analysis_utils
import utils.path_utils as path_utils

reload(plotting_utils)
reload(simulate_markov_utils) 
reload(analysis_utils)
reload(path_utils)

# Re-import the specific functions
from plotting_notebooks.plotting_utils import (
    load_in_combined_results,
    load_monitoring_data,
    plot_markov_analysis,
    plot_roc_curves,
    plot_suspiciousness_distributions,
    plot_cost_vs_safety_curves,
    get_roc_curve,
    interpolate_tpr,
    prepare_monitoring_data,
    merge_monitoring_with_combined_results,
    filter_and_prepare_data,
    is_in_training_data,
    plot_combined_cost_vs_safety_curve,
    plot_combined_cost_vs_tpr_curve,
)
from simulate_markov_utils import plot_markov_df
from utils.analysis_utils import extract_verdict, balance_classes_with_oversampling
from utils.path_utils import resolve_path

In [ ]:
x = combined_results[(combined_results.task_name != 'election_tampering') & (~combined_results.task_name.isin(training_data.task_name.unique())) & (combined_results.monitor_agent == 'claude-3-7-sonnet-20250219')]
fpr, tpr, thresholds = roc_curve(x.transcript_type != 'transcripts_no_secondary_task', x.default_suspiciousness)
auc_score = auc(fpr, tpr)
auc_score

# Force garbage collection
gc.collect()

# Create cost vs safety curves for turn-by-turn monitoring
# Define metrics and their corresponding cost columns
metric_columns = [f'max_suspiciousness_{i}_turn_by_turn' for i in DOWNSAMPLE_FACTORS]
cost_columns = [f'max_suspiciousness_usage_{i}_turn_by_turn' for i in DOWNSAMPLE_FACTORS]

# Add other metrics of interest
# metric_columns += ['final_suspiciousness_turn_by_turn'] + [f'{i}th_highest_suspiciousness_turn_by_turn' for i in [2, 3, 5]]
# cost_columns += ['final_usage_turn_by_turn'] + [f'{i}th_highest_suspiciousness_usage_turn_by_turn' for i in [2, 3, 5]]
metric_columns += ['final_suspiciousness_turn_by_turn'] 
cost_columns += ['final_usage_turn_by_turn'] 
# Add mean_suspiciousness_first_i metrics
metric_columns += [f'mean_suspiciousness_first_{i}' for i in range(1, 11)]
cost_columns += [f'mean_suspiciousness_first_{i}_usage' for i in range(1, 11)]

metric_columns += [f'max_suspiciousness_back_{i}_turn_by_turn' for i in BACK_FRACS[:2]]
cost_columns += [f'max_suspiciousness_back_{i}_usage_turn_by_turn' for i in BACK_FRACS[:2]]

# metric_columns += [f'suspiciousness_{i}' for i in range(1, 11)]
# cost_columns += ['mean_suspiciousness_first_1_usage' for i in range(1, 11)]
# Create a copy of turn_by_turn with renamed monitor_agent values using FT_STEP_MAPPING
turn_by_turn_renamed = turn_by_turn.copy()
ft_sp1_renamed = ft_sp1.copy()
ft_sp5_renamed = ft_sp5.copy()
# Create a reverse mapping from model name to step number for display
def get_display_name(step):
    if step == 0:
        return "Base model"
    if step % 70 == 0:
        return f"Fine-tuned" + ("s" if step//70 > 1 else "")
    if step % 56 == 0:
        return f"Unbalanced data, {step//56} epoch" + ("s" if step//56 > 1 else "")
    return "Base model"
display_names = {model: get_display_name(step) for model, step in FT_STEP_MAPPING.items()}
# Apply the mapping
turn_by_turn_renamed['monitor_display_name'] = turn_by_turn_renamed['monitor_agent'].map(
    lambda x: display_names.get(x, x)
)
ft_sp1_renamed['monitor_display_name'] = ft_sp1_renamed['monitor_agent'].map(
    lambda x: display_names.get(x, x)
)
ft_sp5_renamed['monitor_display_name'] = ft_sp5_renamed['monitor_agent'].map(
    lambda x: display_names.get(x, x)
)
excluded_tasks = list(training_data.task_name.unique()) + ['election_tampering']
# excluded_tasks = [x for x in turn_by_turn_renamed.task_name.unique() if x not in ['american_airlines_2']]
turn_by_turn_renamed = turn_by_turn_renamed[~turn_by_turn_renamed.task_name.isin(excluded_tasks)]
ft_sp1_renamed = ft_sp1_renamed[~ft_sp1_renamed.task_name.isin(excluded_tasks)]
ft_sp5_renamed = ft_sp5_renamed[~ft_sp5_renamed.task_name.isin(excluded_tasks)]

target_size = 20
ft_sp1_renamed = pd.concat([balance_classes_with_oversampling(ft_sp1_renamed[(ft_sp1_renamed.transcript_type == x) & (ft_sp1_renamed.monitor_agent == y)], target_size=target_size) for x in ft_sp1_renamed.transcript_type.unique() for y in ft_sp1_renamed.monitor_agent.unique()])
ft_sp5_renamed = pd.concat([balance_classes_with_oversampling(ft_sp5_renamed[(ft_sp5_renamed.transcript_type == x) & (ft_sp5_renamed.monitor_agent == y)], target_size=target_size) for x in ft_sp5_renamed.transcript_type.unique() for y in ft_sp5_renamed.monitor_agent.unique()])
turn_by_turn_renamed = pd.concat([balance_classes_with_oversampling(turn_by_turn_renamed[(turn_by_turn_renamed.transcript_type == x) & (turn_by_turn_renamed.monitor_agent == y)], target_size=target_size) for x in turn_by_turn_renamed.transcript_type.unique() for y in turn_by_turn_renamed.monitor_agent.unique()])
ft_sp1_renamed = ft_sp1_renamed[ft_sp1_renamed.task_name.isin([x for x in ft_sp1_renamed.task_name.unique() if len(ft_sp1_renamed[ft_sp1_renamed.task_name == x]) > target_size])]
ft_sp5_renamed = ft_sp5_renamed[ft_sp5_renamed.task_name.isin([x for x in ft_sp5_renamed.task_name.unique() if len(ft_sp5_renamed[ft_sp5_renamed.task_name == x]) > target_size])]
turn_by_turn_renamed = turn_by_turn_renamed[turn_by_turn_renamed.task_name.isin([x for x in turn_by_turn_renamed.task_name.unique() if len(turn_by_turn_renamed[turn_by_turn_renamed.task_name == x]) > target_size])]
ft_sp1_renamed = balance_classes_with_oversampling(ft_sp1_renamed, target_size=3 * target_size)
ft_sp5_renamed = balance_classes_with_oversampling(ft_sp5_renamed, target_size=3 * target_size)
turn_by_turn_renamed = balance_classes_with_oversampling(turn_by_turn_renamed, target_size=3 * target_size)
# Use the new utility functions to create the plots
# Plot cost vs safety curve (AUC)
# Reload plotting_utils module
import importlib
import plotting_notebooks.plotting_utils
importlib.reload(plotting_notebooks.plotting_utils)
from plotting_notebooks.plotting_utils import plot_combined_cost_vs_safety_curve, plot_combined_cost_vs_tpr_curve

performance_df = plot_combined_cost_vs_safety_curve(
    ft_sp1_df=ft_sp1_renamed,
    ft_sp5_df=ft_sp5_renamed,
    turn_by_turn_df=turn_by_turn_renamed,
    metric_columns=metric_columns,
    cost_columns=cost_columns,
    figsize=(10, 6)
)

In [ ]:
# Clear memory
gc.collect()
print_memory_usage()

# Plot cost vs TPR curve
tpr_performance_df = plot_combined_cost_vs_tpr_curve(
    ft_sp1_df=ft_sp1_renamed,
    ft_sp5_df=ft_sp5_renamed,
    turn_by_turn_df=turn_by_turn_renamed,
    metric_columns=metric_columns,
    cost_columns=cost_columns,
    target_fpr=0.01
)

# Clear memory
gc.collect()
print_memory_usage()

In [ ]:
ft_sp1_renamed[ft_sp1_renamed.monitor_agent == 'gpt-4o-2024-08-06'].task_name.value_counts()


In [ ]:
counts = {}
for monitor in ft_sp1_renamed.monitor_agent.unique():
    for i, row in ft_sp1_renamed[ft_sp1_renamed.monitor_agent == monitor].iterrows():
        key = (row.task_name, row.transcript_type, row.file_name)
        if key not in counts:
            counts[key] = 0
        counts[key] += 1

In [ ]:
for monitor in ft_sp1_renamed.monitor_agent.unique():
    for i, row in ft_sp1_renamed[ft_sp1_renamed.monitor_agent == monitor].iterrows():
        key = (row.task_name, row.transcript_type, row.file_name)
        if counts[key] == 3:
            display(row)
            5/0

In [ ]:
ft_sp1_renamed[(ft_sp1_renamed.task_name == 'notes_summarization') & (ft_sp1_renamed.transcript_type == 'transcripts')]

In [ ]:
combined_results[(combined_results.task_name == 'notes_summarization') & (combined_results.transcript_type == 'transcripts')]